In [ ]:
from squiggs.neuron_viewer import NeuronViewer
from squiggs.renderers import FitRenderer
from sg.eval_models import plot_summary

import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt
from pathlib import Path
from utils.paths import PROJECT_ROOT
import pickle

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
# gs over regularization constant

In [ ]:
subj_id = "MR82"
sess_id = "20251027_152036"
region = "DMS"
m = 0
a = 1

with open(
    PROJECT_ROOT.parents[0]
    / f"vars/families/{subj_id}/{sess_id}/no_pr/{region}/family-m{m}a{a}.pkl",
    "rb",
) as f:
    family = pickle.load(f)

In [ ]:
from squiggs.renderers import PETHRasterRenderer
from core.data import get_psths_cond, get_choice_ts
from utils.paths import FIGURES_DIR

mode = "response"

renderer = PETHRasterRenderer(
    event_times=get_choice_ts(family.trial_data, mode=mode),
    spike_times=family.spike_times[region],
    peths=get_psths_cond(family.psths[region], family.trial_data, mode=mode),
    pres=0.5,
    posts=1,
    binwidth_s=25 / 1000,
    s=0.2,
    linewidths=0.2,
    save_subdir=Path("peths") / family.subj_id / family.sess_id / region / mode,
)

nv = NeuronViewer(
    num_units=family.psths[region].shape[0], render_func=renderer, fig_dir=FIGURES_DIR
)

In [ ]:
import torch

torch.mean(family.res_affine["r2test"])

In [ ]:
renderer = FitRenderer(
    family.mod_affine,
    x=family.test_dl.dataset[:],
    y=family.robs,
    save_subdir=Path("model_fits") / "0312-lm" / "offset",
)

nv = NeuronViewer(num_units=renderer.y.shape[1], render_func=renderer)

In [ ]:
plot_summary(family, model=family.mod_offset, potato=family.strategy, mode="offset")

In [ ]:
from sg.eval_models import get_latent_r

max_n_latents = 4


def get_r2s_helper(subj_id, sess_id, regions):
    r2s = {}
    for reg in regions:
        r2s[reg] = get_latent_r(
            subj_id,
            sess_id,
            n_m=max_n_latents,
            n_a=max_n_latents,
            region=reg,
            do_plot=True,
            do_save=True,
        )
    return r2s


subj_id = "MM012"
sess_id = "20231219_130847"
r2s = get_r2s_helper(
    subj_id,
    sess_id,
    regions=["all", "ACC", "M2", "DLS", "DMS"],
)

In [ ]:
subj_id = "MM012"
sess_id = "20231215_140148"
r2s = get_r2s_helper(
    subj_id,
    sess_id,
    regions=["all", "ACC", "M2", "DLS", "DMS"],
)

In [ ]:
subj_id = "MR82"
sess_id = "20251027_152036"
r2s = get_r2s_helper(
    subj_id,
    sess_id,
    regions=["all", "ACC", "M2", "DLS", "DMS"],
)

In [ ]:
subj_id = "MR83"
sess_id = "20251027_162326"
r2s = get_r2s_helper(
    subj_id,
    sess_id,
    regions=["all", "ACC", "M2", "DLS", "DMS"],
)

In [ ]:
# two big things
# one is the addition of population rate
# the second is the time window (0.5 to 0.5 -> 0.5 to 1 -> 0.5 to 0.5)